# Week 91, Day 3: Error handling and logging

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w091d3.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
def parse_km(text):
    try:
        return float(text)
    except Exception:
        return 0.0

print([parse_km(t) for t in ["6.5", "6,5", "six"]])

> **Key idea.** Raise specific errors from one family that carry a status, a code and a user message; translate at the edge, and treat anything else as a bug: logged in full, reported to the user only with a reference. Catch narrowly. Keep causes with `raise ... from` and add context with `add_note`. Log JSON records with levels, a request id and arguments rather than f-strings. Retry only transient failures, a limited number of times, with exponential backoff.

### Exercise: Errors with a status

Write `WaymarkError(Exception)` with class attributes `status = 400` and `code = "bad_request"`, and subclasses `UnknownPlaceError` (`404`, `"unknown_place"`), `NoRouteError` (`422`, `"no_route"`) and `BadInput` (`400`, `"bad_input"`). Then `to_response(exc, new_reference)`: for a `WaymarkError`, return `(status, {"error": {"code": CODE, "message": str(exc)}})`; for anything else, `(500, {"error": {"code": "internal", "message": "Something went wrong. Quote reference REF if you report it."}})`, where REF is `new_reference()`.

In [ ]:
class WaymarkError(Exception):
    ___

def to_response(exc, new_reference):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_family():\n    \"one family\"\n    assert issubclass(UnknownPlaceError, WaymarkError) and issubclass(NoRouteError, WaymarkError) and issubclass(BadInput, WaymarkError)\n    assert not issubclass(WaymarkError, LookupError)\n\ndef test_responses():\n    \"statuses and messages at the edge\"\n    ref = lambda: \"e-7f3a\"\n    assert to_response(UnknownPlaceError(\"No junction called 'Ridgee'\"), ref) == (404, {\"error\": {\"code\": \"unknown_place\", \"message\": \"No junction called 'Ridgee'\"}})\n    assert to_response(NoRouteError(\"No route from Mill to Island\"), ref)[0] == 422\n    assert to_response(BadInput(\"km must be positive\"), ref)[1][\"error\"][\"code\"] == \"bad_input\"\n    status, body = to_response(KeyError(\"internal_cache_key\"), ref)\n    assert status == 500 and body == {\"error\": {\"code\": \"internal\", \"message\": \"Something went wrong. Quote reference e-7f3a if you report it.\"}}\n    assert \"internal_cache_key\" not in str(body)")

### Exercise: Which line?

Write `load_walks(lines)`: each line is `day,km,minutes`; return a list of dicts with `day`, `km` (float) and `minutes` (int). If converting a line raises `ValueError`, add the note `"line N: LINE"` (N counting from 1, LINE written with `repr`) to that same error and re-raise it.

In [ ]:
def load_walks(lines):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_ok():\n    \"good lines\"\n    assert load_walks([\"2026-01-04,6.0,105\"]) == [{\"day\": \"2026-01-04\", \"km\": 6.0, \"minutes\": 105}]\n\ndef test_note():\n    \"the same error, with context\"\n    try:\n        load_walks([\"2026-01-04,6.0,105\", \"2026-01-18,seven,98\"])\n    except ValueError as e:\n        assert \"seven\" in str(e), \"the original error, not a new one\"\n        assert e.__notes__ == [\"line 2: '2026-01-18,seven,98'\"]\n    else:\n        raise AssertionError(\"no error\")")

### Exercise: Logs as JSON

Write `JsonFormatter(logging.Formatter)` whose `format(record)` returns one line of JSON with the keys `level` (the level name), `logger`, `message` (the message with its arguments filled in), then any of `request_id`, `user` and `duration_ms` the record has (passed with `extra=`), in that order, and, if the record carries exception information, `error` as `"TYPE: MESSAGE"`.

In [ ]:
import json
import logging

class JsonFormatter(logging.Formatter):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import io, json, logging\n\ndef capture():\n    stream = io.StringIO()\n    logger = logging.getLogger(\"waymark.check\")\n    logger.handlers.clear()\n    handler = logging.StreamHandler(stream)\n    handler.setFormatter(JsonFormatter())\n    logger.addHandler(handler)\n    logger.setLevel(logging.INFO)\n    logger.propagate = False\n    return logger, stream\n\ndef test_records():\n    \"one JSON object per line\"\n    logger, stream = capture()\n    logger.info(\"planned %s to %s\", \"Mill\", \"Ridge\", extra={\"request_id\": \"r1\", \"duration_ms\": 4.2})\n    logger.debug(\"not shown\")\n    logger.warning(\"slow plan\", extra={\"user\": \"ada\", \"request_id\": \"r2\"})\n    lines = stream.getvalue().splitlines()\n    assert len(lines) == 2\n    assert lines[0] == json.dumps({\"level\": \"INFO\", \"logger\": \"waymark.check\", \"message\": \"planned Mill to Ridge\", \"request_id\": \"r1\", \"duration_ms\": 4.2})\n    assert list(json.loads(lines[1])) == [\"level\", \"logger\", \"message\", \"request_id\", \"user\"]\n\ndef test_exception():\n    \"errors, summarised\"\n    logger, stream = capture()\n    try:\n        1 / 0\n    except ZeroDivisionError:\n        logger.exception(\"pace failed\", extra={\"request_id\": \"r3\"})\n    record = json.loads(stream.getvalue())\n    assert record[\"level\"] == \"ERROR\" and record[\"error\"] == \"ZeroDivisionError: division by zero\"")

### Exercise: Retrying with backoff

Write `retry(fn, attempts=3, base=0.5, retry_on=(TimeoutError,), sleep=time.sleep)`: call `fn()` and return its result. If it raises one of `retry_on`, sleep for `base * 2 ** i` seconds (i = 0, 1, ...) and try again, up to `attempts` calls in all; if the last attempt fails, re-raise its error. Any other error is raised at once, without retrying.

In [ ]:
import time

def retry(fn, attempts=3, base=0.5, retry_on=(TimeoutError,), sleep=time.sleep):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def flaky(failures, error=TimeoutError):\n    calls = []\n    def fn():\n        calls.append(1)\n        if len(calls) <= failures:\n            raise error(\"busy\")\n        return \"ok\"\n    return fn, calls\n\ndef test_recovers():\n    \"transient failures\"\n    fn, calls = flaky(2)\n    slept = []\n    assert retry(fn, sleep=slept.append) == \"ok\" and len(calls) == 3 and slept == [0.5, 1.0]\n\ndef test_gives_up():\n    \"a limited number of attempts\"\n    fn, calls = flaky(10)\n    slept = []\n    try:\n        retry(fn, attempts=4, base=1, sleep=slept.append)\n    except TimeoutError:\n        assert len(calls) == 4 and slept == [1, 2, 4]\n    else:\n        raise AssertionError(\"should give up\")\n\ndef test_not_transient():\n    \"bad input fails at once\"\n    fn, calls = flaky(1, ValueError)\n    slept = []\n    try:\n        retry(fn, sleep=slept.append)\n    except ValueError:\n        assert len(calls) == 1 and slept == []\n    else:\n        raise AssertionError(\"ValueError\")")

### Bug hunt: Zero-kilometre walks

`parse_km` swallows every error, so `"6,5"` becomes a walk of 0 km and the program prints `0.0`. Let only `ValueError` through as a `BadInput` with the message `'6,5' is not a distance in km`, so it prints `error: '6,5' is not a distance in km`.

In [ ]:
class BadInput(Exception):
    pass

def parse_km(text):
    try:
        return float(text)
    except Exception:
        return 0.0

try:
    print(parse_km("6,5"))
except BadInput as e:
    print("error:", e)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"fail loudly\"\n    expect_output(\"error: '6,5' is not a distance in km\")")

### Bug hunt: A cause thrown away

`load_map` translates the error but loses the original, so the program prints `NoneType`. Keep the cause, so it prints `FileNotFoundError`.

In [ ]:
class WaymarkError(Exception):
    pass

def load_map(path):
    try:
        with open(path) as f:
            return f.read()
    except FileNotFoundError:
        raise WaymarkError(f"no map at {path}")

try:
    load_map("missing-map.csv")
except WaymarkError as e:
    print(type(e.__cause__).__name__)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"raise from\"\n    expect_output(\"FileNotFoundError\")")

### Puzzle: Retrying

Arrange `retry` so it tries up to `attempts` times, re-raises on the last failure, and backs off between tries. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # logging.WARNING as a number
q = ___   # the level for "a route was planned" (a string)
r = ___   # the sleeps retry makes with base 0.5 and 4 attempts that all fail (a list)
s = ___   # the status of a NoRouteError in today's hierarchy
t = ___   # True or False: an unexpected error's message should be shown to the user

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 30 and q == \"INFO\" and r == [0.5, 1.0, 2.0], \"p, q and r\"\n    assert s == 422 and t is False, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: what broke today?

Write `incident_report(lines)`. Each line should be a JSON object with at least `level` and `message`, perhaps with `logger`, `request_id` and `duration_ms`; anything else (not JSON, not an object, or missing either key) is **unparsed**. Return a dict with:

- `"levels"`: counts per level for the levels present, in the order DEBUG, INFO, WARNING, ERROR, CRITICAL;
- `"unparsed"`: the number of unparsed lines;
- `"failed"`: for each request id (in order of first appearance; records without one use `"-"`) that has an ERROR or CRITICAL record, `"ID: MESSAGE (LOGGER)"` for its first such record (`LOGGER` is `"?"` if missing);
- `"slowest"`: the three records with the largest `duration_ms` (ties by request id), as `"ID: D ms"`, with D as it appears in the record.

In [ ]:
import json

def incident_report(lines):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("LOG = [\n    '{\"level\": \"INFO\", \"logger\": \"waymark.api\", \"message\": \"GET /v1/plan\", \"request_id\": \"r1\", \"duration_ms\": 4.2}',\n    '{\"level\": \"INFO\", \"logger\": \"waymark.api\", \"message\": \"GET /v1/walks\", \"request_id\": \"r2\", \"duration_ms\": 120}',\n    '{\"level\": \"WARNING\", \"logger\": \"waymark.planner\", \"message\": \"slow search\", \"request_id\": \"r2\"}',\n    'Traceback (most recent call last):',\n    '{\"level\": \"ERROR\", \"logger\": \"waymark.share\", \"message\": \"share store unavailable\", \"request_id\": \"r3\"}',\n    '{\"level\": \"ERROR\", \"logger\": \"waymark.share\", \"message\": \"retry failed\", \"request_id\": \"r3\", \"duration_ms\": 2050}',\n    '{\"level\": \"CRITICAL\", \"message\": \"database connection lost\"}',\n    '[\"not\", \"an\", \"object\"]',\n    '{\"level\": \"INFO\", \"logger\": \"waymark.api\", \"message\": \"GET /v1/plan\", \"request_id\": \"r4\", \"duration_ms\": 120}',\n    '{\"level\": \"DEBUG\", \"logger\": \"waymark.planner\", \"message\": \"heap size 12\", \"request_id\": \"r1\"}',\n    '{\"message\": \"no level\"}',\n]\n\ndef test_report():\n    \"a day of logs\"\n    report = incident_report(LOG)\n    assert report[\"levels\"] == {\"DEBUG\": 1, \"INFO\": 3, \"WARNING\": 1, \"ERROR\": 2, \"CRITICAL\": 1}\n    assert list(report[\"levels\"]) == [\"DEBUG\", \"INFO\", \"WARNING\", \"ERROR\", \"CRITICAL\"]\n    assert report[\"unparsed\"] == 3\n    assert report[\"failed\"] == [\"r3: share store unavailable (waymark.share)\", \"-: database connection lost (?)\"]\n    assert report[\"slowest\"] == [\"r3: 2050 ms\", \"r2: 120 ms\", \"r4: 120 ms\"]\n\ndef test_quiet():\n    \"a quiet day\"\n    assert incident_report(['{\"level\": \"INFO\", \"message\": \"ok\"}']) == {\"levels\": {\"INFO\": 1}, \"unparsed\": 0, \"failed\": [], \"slowest\": []}")

## Warm-up

Two typing mistakes became walks of 0 km, silently. Every statistic is now wrong and nothing says why. An error that's hidden doesn't go away: it turns up later, somewhere else, harder to explain.

## Errors users can act on

Inside the program, errors should be **specific**: `UnknownPlaceError`, `NoRouteError`, `BadInput`, all subclasses of one `WaymarkError`, so callers can catch the family or one member. Each carries what the **edge** needs: an HTTP status, a short machine-readable `code`, and a message written for the user ("No junction called 'Ridgee'. Did you mean Ridge?").

At the **edge** (the API handler, the command line's `main`), translate:

- a `WaymarkError` becomes its status and message (or, on the command line, a message on standard error and exit status 1);
- anything else is a **bug**: log it in full, and tell the user only that something went wrong, with a **reference** they can quote. Never send a traceback or an internal message to a user: it confuses them and helps attackers.

Catch **narrowly**: `except ValueError` around the one line that can raise it, not `except Exception` around a page of code.

## Keeping the cause

When you translate an error, keep the original: `raise BadInput("map file not found") from err` sets `__cause__`, and the traceback shows both. And when an error passes through a place that knows more (which line of a file, which junction), add it with `err.add_note("line 2 of walks.csv")` and re-raise the same error with a bare `raise`: the note is printed with the traceback and kept in `err.__notes__`.

## Structured logs

`logging` gives each message a **level**: `DEBUG` (details for a developer), `INFO` (normal events: a route was planned), `WARNING` (something unexpected, handled), `ERROR` (something failed), `CRITICAL` (the service can't continue). Use `logger = logging.getLogger("waymark.planner")` per module and configure handlers once, at the edge.

Logs are read by machines first: searched, counted and alerted on. So write each record as one **JSON** object with fixed fields (`level`, `logger`, `message`) plus context passed with `extra=`: a **request id** that ties together every record from one request, the user, the duration. Pass values as arguments (`logger.info("planned %s to %s", a, b)`), not f-strings, so records with the same template can be grouped. And remember Week 90: secrets never reach a log.

## Retrying

Some failures are **transient**: a timeout, a busy server. Retrying those, after a growing delay (**exponential backoff**: 0.5 s, 1 s, 2 s), often works. Retry only those: a `BadInput` will fail the same way every time. Limit the attempts, add a little random **jitter** in production so many clients don't retry in step, and only retry operations that are safe to repeat (Week 90's idempotency).

## Practice

## Quiz

## Daily challenge

## Your capstone today

1. **An error family** for your capstone, with statuses or exit codes, and one place at the edge that translates them.
2. **Hunt the swallowers**: search for `except Exception` and bare `except:`, and narrow or remove each one.
3. **Structured logs** with a request id (or a run id for a command-line tool), and no secrets in them.
4. **Break it on purpose**: wrong input, a missing file, a dead network. Check every failure gives a useful message and a useful log record.

## Before you go

Tomorrow is documentation: the README, the reference, and docstrings that are checked so they can't drift from the code.